# Notebook 28. Single-Event 3-D Case Study (Preserved Prototype)

This notebook prototypes a **single-event 3-D case-study viewer** for one February 2018 event.

The first-pass goal is simple and readable visualization, not every possible field at once. The notebook focuses on:

- terrain as the lower boundary of the cube
- an upper-level jet rendered as core points by default, with an optional experimental volume mode
- a derived jet-axis line with automatic entrance and exit jet-normal slices
- smoothed jet-normal omega curtains to show ascent and descent through the jet environment
- strongest low-level convergence over the terrain, with optional moist-volume experiments kept secondary
- a companion pressure-coordinate cross section using the same slice endpoints

Default prototype target:

- peak time: `2018-02-03 20:00 UTC`
- event family: the stronger February 2018 case inside the `2018-02-02` to `2018-02-07` window

To change the case or the slice, edit the constants in the setup cell below and rerun from there.


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git"
BRANCH = os.environ.get("JPCZ_CATALOG_BRANCH", "codex/notebook16-pcolormesh")
REPO_DIR = "/content/JPCZcatalog"
FORCE_REFRESH_REPO = False
PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/JPCZcatalog_outputs"

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
    print("Persistent output dir:", DRIVE_OUTPUT_DIR)

os.chdir("/content")


def clone_repo_branch():
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR],
        text=True,
        capture_output=True,
    )
    print(proc.stdout)
    print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{proc.stderr}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
        check=True,
    )


def sync_repo_branch():
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)


if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("Removed existing repo clone:", REPO_DIR)

if not os.path.exists(REPO_DIR):
    clone_repo_branch()
else:
    print("Using existing repo clone:", REPO_DIR)

try:
    sync_repo_branch()
except subprocess.CalledProcessError:
    print("Existing clone could not switch branches cleanly. Re-cloning target branch.")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    clone_repo_branch()
    sync_repo_branch()

os.chdir(REPO_DIR)
src_dir = os.path.join(REPO_DIR, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

active_branch = subprocess.run(["git", "-C", REPO_DIR, "branch", "--show-current"], text=True, capture_output=True, check=True).stdout.strip()
print("Working directory:", os.getcwd())
print("Runtime repo branch:", active_branch)


In [ ]:
from pathlib import Path
import importlib

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
import plotly.io as pio

from jpcz_catalog.config import BoundingBox
from jpcz_catalog.diagnostics import load_snapshot, compute_geopotential_height_field, compute_wind_speed_field
from jpcz_catalog.era5 import open_arco_era5
import jpcz_catalog.cross_sections as cross_sections_module
import jpcz_catalog.three_d_case_study as three_d_case_study_module

cross_sections_module = importlib.reload(cross_sections_module)
three_d_case_study_module = importlib.reload(three_d_case_study_module)

build_3d_case_study_data = three_d_case_study_module.build_3d_case_study_data
build_case_metadata_table = three_d_case_study_module.build_case_metadata_table
build_case_runtime_diagnostics = three_d_case_study_module.build_case_runtime_diagnostics
build_jet_slice_table = three_d_case_study_module.build_jet_slice_table
case_with_slice = three_d_case_study_module.case_with_slice
create_3d_case_figure = three_d_case_study_module.create_3d_case_figure
derive_jet_slice_guide = three_d_case_study_module.derive_jet_slice_guide
load_case_event_catalog = three_d_case_study_module.load_case_event_catalog
load_surface_elevation_field = three_d_case_study_module.load_surface_elevation_field
select_case_event = three_d_case_study_module.select_case_event

pio.renderers.default = "colab" if "google.colab" in sys.modules else "notebook_connected"

CASE_EVENT_PEAK_UTC = "2018-02-03 20:00:00"
CASE_EVENT_OFFSET_HOURS = 0
CASE_DOMAIN = BoundingBox(lon_min=126.0, lon_max=144.0, lat_min=34.0, lat_max=46.0)
USE_JET_RELATIVE_SLICES = True
JET_SLICE_LEVEL_HPA = 300
JET_SLICE_THRESHOLD_MS = 35.0
SLICE_START = (130.5, 36.0)
SLICE_END = (141.5, 42.0)
CASE_CATALOG_PATH = Path("outputs/verification/jpcz_catalog_ndjf_merged_12h_manual_verification.csv")
SURFACE_ELEVATION_CACHE_PATH = Path("outputs/verification/objective_subtype_t850_elevation_mask_sensitivity/objective_subtype_surface_elevation.nc")
SURFACE_ELEVATION_DRIVE_FALLBACK_PATH = Path(DRIVE_OUTPUT_DIR) / "objective_subtype_surface_elevation.nc"
CASE_PLOT_DIR = Path("outputs/verification/single_event_3d_case_study_plots")
CASE_PLOT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PLOTS = False

JET_CONTEXT_DOMAIN = BoundingBox(lon_min=118.0, lon_max=152.0, lat_min=28.0, lat_max=50.0)
JET_ISOTACH_MIN_WIND_SPEED = 30.0
JET_ISOTACH_LEVELS = np.arange(30.0, 90.0 + 5.0, 5.0)

ERA5_RUNTIME_CACHE_27 = None


def get_era5_runtime_ds_27():
    global ERA5_RUNTIME_CACHE_27
    if ERA5_RUNTIME_CACHE_27 is None:
        ERA5_RUNTIME_CACHE_27 = open_arco_era5(chunks={"time": 24})
    return ERA5_RUNTIME_CACHE_27


def rounded_contour_levels(field, step):
    values = np.asarray(field.values, dtype=float)
    values = values[np.isfinite(values)]
    lower = np.floor(np.nanmin(values) / step) * step
    upper = np.ceil(np.nanmax(values) / step) * step
    if lower == upper:
        upper = lower + step
    return np.arange(lower, upper + 0.5 * step, step)


def configure_case_axis(ax, domain, title):
    ax.set_extent([domain.lon_min, domain.lon_max, domain.lat_min, domain.lat_max], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="#efefef", edgecolor="none", zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=0.7, edgecolor="black")
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.30, edgecolor="black")
    gl = ax.gridlines(draw_labels=True, linestyle="--", linewidth=0.25, alpha=0.35, color="0.65")
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {"size": 8}
    gl.ylabel_style = {"size": 8}
    ax.set_title(title, fontsize=11, loc="left")


def plot_case_overview_map(analysis_time, slice_start=None, slice_end=None, jet_slice_guide=None):
    era5_runtime_ds = get_era5_runtime_ds_27()
    snapshot_300 = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["u_component_of_wind", "v_component_of_wind", "geopotential"],
        domain=JET_CONTEXT_DOMAIN,
        level=300,
    )
    snapshot_500 = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["geopotential"],
        domain=JET_CONTEXT_DOMAIN,
        level=500,
    )
    msl_snapshot = load_snapshot(
        era5_runtime_ds,
        analysis_time,
        variables=["mean_sea_level_pressure"],
        domain=JET_CONTEXT_DOMAIN,
    )

    wind_speed_300 = compute_wind_speed_field(snapshot_300)
    jet_isotach_field = wind_speed_300.where(wind_speed_300 >= JET_ISOTACH_MIN_WIND_SPEED)
    z500 = compute_geopotential_height_field(snapshot_500)
    msl_hpa = (msl_snapshot["mean_sea_level_pressure"] / 100.0).rename("msl_hpa")

    fig, ax = plt.subplots(figsize=(10.8, 6.8), subplot_kw={"projection": ccrs.PlateCarree()})
    fill = ax.contourf(
        jet_isotach_field.longitude,
        jet_isotach_field.latitude,
        jet_isotach_field,
        levels=JET_ISOTACH_LEVELS,
        cmap="YlGnBu_r",
        extend="max",
        transform=ccrs.PlateCarree(),
    )
    configure_case_axis(ax, JET_CONTEXT_DOMAIN, title="Overview map: 300 hPa jet isotachs + 500 hPa height + surface MSLP")
    z500_contours = ax.contour(
        z500.longitude,
        z500.latitude,
        z500,
        levels=rounded_contour_levels(z500, 60.0),
        colors="#303030",
        linewidths=0.9,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(z500_contours, inline=True, fontsize=7, fmt="%d")
    msl_contours = ax.contour(
        msl_hpa.longitude,
        msl_hpa.latitude,
        msl_hpa,
        levels=rounded_contour_levels(msl_hpa, 4.0),
        colors="#39ff14",
        linewidths=0.9,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(msl_contours, inline=True, fontsize=6.5, fmt="%d")

    if jet_slice_guide is not None:
        ax.plot([jet_slice_guide.axis_start[0], jet_slice_guide.axis_end[0]], [jet_slice_guide.axis_start[1], jet_slice_guide.axis_end[1]], color="#111827", linewidth=2.6, linestyle="--", transform=ccrs.PlateCarree(), zorder=7)
        ax.plot([jet_slice_guide.entrance_slice_start[0], jet_slice_guide.entrance_slice_end[0]], [jet_slice_guide.entrance_slice_start[1], jet_slice_guide.entrance_slice_end[1]], color="#7c3aed", linewidth=2.6, transform=ccrs.PlateCarree(), zorder=8)
        ax.plot([jet_slice_guide.exit_slice_start[0], jet_slice_guide.exit_slice_end[0]], [jet_slice_guide.exit_slice_start[1], jet_slice_guide.exit_slice_end[1]], color="#ea580c", linewidth=2.6, transform=ccrs.PlateCarree(), zorder=8)
        entrance_mid_lon = 0.5 * (jet_slice_guide.entrance_slice_start[0] + jet_slice_guide.entrance_slice_end[0])
        entrance_mid_lat = 0.5 * (jet_slice_guide.entrance_slice_start[1] + jet_slice_guide.entrance_slice_end[1])
        exit_mid_lon = 0.5 * (jet_slice_guide.exit_slice_start[0] + jet_slice_guide.exit_slice_end[0])
        exit_mid_lat = 0.5 * (jet_slice_guide.exit_slice_start[1] + jet_slice_guide.exit_slice_end[1])
        ax.text(entrance_mid_lon, entrance_mid_lat + 0.5, "Entrance slice", color="#6d28d9", fontsize=8, transform=ccrs.PlateCarree(), ha="center", va="bottom")
        ax.text(exit_mid_lon, exit_mid_lat + 0.5, "Exit slice", color="#9a3412", fontsize=8, transform=ccrs.PlateCarree(), ha="center", va="bottom")
    elif slice_start is not None and slice_end is not None:
        ax.plot([slice_start[0], slice_end[0]], [slice_start[1], slice_end[1]], color="#7c3aed", linewidth=2.4, transform=ccrs.PlateCarree(), zorder=7)
        ax.scatter([slice_start[0]], [slice_start[1]], color="#16a34a", s=55, transform=ccrs.PlateCarree(), zorder=8)
        ax.scatter([slice_end[0]], [slice_end[1]], color="#dc2626", s=55, transform=ccrs.PlateCarree(), zorder=8)
        ax.text(slice_start[0], slice_start[1] - 0.6, "Start", color="#166534", fontsize=8, transform=ccrs.PlateCarree(), ha="center", va="top")
        ax.text(slice_end[0], slice_end[1] + 0.6, "End", color="#991b1b", fontsize=8, transform=ccrs.PlateCarree(), ha="center", va="bottom")

    cbar = fig.colorbar(fill, ax=ax, orientation="horizontal", pad=0.07, aspect=38)
    cbar.set_label(f"300 hPa wind speed [m s^-1] (only >= {JET_ISOTACH_MIN_WIND_SPEED:.0f} shown)")
    return fig


def maybe_save_case_output(fig, filename):
    if not SAVE_PLOTS:
        return None
    output_path = CASE_PLOT_DIR / filename
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    return output_path


print("Notebook 28 helpers are ready")
print("Default case peak:", CASE_EVENT_PEAK_UTC)
print("Jet-relative slices enabled:", USE_JET_RELATIVE_SLICES)
print("Manual fallback slice:", SLICE_START, "->", SLICE_END)


In [ ]:
catalog_df = load_case_event_catalog(CASE_CATALOG_PATH)
february_window_df = catalog_df.loc[
    (catalog_df["event_peak"] >= pd.Timestamp("2018-02-02 00:00:00"))
    & (catalog_df["event_peak"] <= pd.Timestamp("2018-02-07 23:00:00"))
].copy()
display(Markdown("## Nearby February 2018 candidate events"))
candidate_display_columns = [
    column_name
    for column_name in [
        "event_start",
        "event_end",
        "event_peak",
        "monsoon_type",
        "shinoda_class",
        "candidate_peak_convergence_1e5_s-1",
        "verified_event",
        "verification_notes",
    ]
    if column_name in february_window_df.columns
]
display(february_window_df[candidate_display_columns].reset_index(drop=True))

case_event_row = select_case_event(catalog_df, peak_time_utc=CASE_EVENT_PEAK_UTC)
analysis_time = pd.Timestamp(CASE_EVENT_PEAK_UTC) + pd.Timedelta(hours=CASE_EVENT_OFFSET_HOURS)
display(Markdown("## Selected event summary"))
display(build_case_metadata_table(case_event_row))
print("Analysis time used for the 3-D cube:", analysis_time)


In [ ]:
terrain_cache_path = SURFACE_ELEVATION_CACHE_PATH if SURFACE_ELEVATION_CACHE_PATH.exists() else SURFACE_ELEVATION_DRIVE_FALLBACK_PATH
terrain_field = load_surface_elevation_field(terrain_cache_path, domain=CASE_DOMAIN)
case_data = build_3d_case_study_data(
    get_era5_runtime_ds_27(),
    analysis_time,
    domain=CASE_DOMAIN,
    terrain_field=terrain_field,
    slice_start=None if USE_JET_RELATIVE_SLICES else SLICE_START,
    slice_end=None if USE_JET_RELATIVE_SLICES else SLICE_END,
)
jet_slice_guide = None
entrance_case_data = case_data
exit_case_data = None
primary_slice_start = SLICE_START
primary_slice_end = SLICE_END
primary_slice_label = "manual"
if USE_JET_RELATIVE_SLICES:
    jet_slice_guide = derive_jet_slice_guide(
        case_data,
        level_hpa=JET_SLICE_LEVEL_HPA,
        jet_threshold_ms=JET_SLICE_THRESHOLD_MS,
    )
    display(Markdown("## Derived jet-relative slices"))
    display(build_jet_slice_table(jet_slice_guide))
    entrance_case_data = case_with_slice(
        case_data,
        slice_start=jet_slice_guide.entrance_slice_start,
        slice_end=jet_slice_guide.entrance_slice_end,
    )
    exit_case_data = case_with_slice(
        case_data,
        slice_start=jet_slice_guide.exit_slice_start,
        slice_end=jet_slice_guide.exit_slice_end,
    )
    primary_slice_start = jet_slice_guide.entrance_slice_start
    primary_slice_end = jet_slice_guide.entrance_slice_end
    primary_slice_label = "entrance"
else:
    entrance_case_data = case_with_slice(case_data, slice_start=SLICE_START, slice_end=SLICE_END)

print("Loaded 3-D case-study volume")
print("Levels:", list(case_data.pressure_volume.level.values.astype(int)))
print("Domain:", CASE_DOMAIN)
print("Terrain cache path used:", terrain_cache_path)
display(Markdown("## Runtime diagnostics"))
display(build_case_runtime_diagnostics(case_data))


In [ ]:
overview_fig = plot_case_overview_map(
    analysis_time,
    slice_start=None if USE_JET_RELATIVE_SLICES else SLICE_START,
    slice_end=None if USE_JET_RELATIVE_SLICES else SLICE_END,
    jet_slice_guide=jet_slice_guide,
)
display(overview_fig)
maybe_save_case_output(overview_fig, f"case27_overview_{analysis_time:%Y%m%d_%H%M}.png")


In [ ]:
JET_RENDER_MODE = "points"  # choose from: "points", "volume", "both"
JET_SLICE_MODE = "auto_entrance_exit" if USE_JET_RELATIVE_SLICES else "manual"

jet_context_fig = create_3d_case_figure(
    case_data,
    title=(
        f"Notebook 28 | jet context cube | {analysis_time:%Y-%m-%d %H:%M UTC}"
        f" | mode={JET_RENDER_MODE} | slices={JET_SLICE_MODE}"
    ),
    show_cube_frame=True,
    show_jet_volume=JET_RENDER_MODE in {"volume", "both"},
    show_jet_points=JET_RENDER_MODE in {"points", "both"},
    jet_isomin=28.0,
    jet_surface_count=14,
    jet_opacity=0.22,
    jet_top_pressure_hpa=400,
    jet_point_threshold=32.0,
    jet_point_size=3.4,
    jet_point_opacity=0.82,
    show_moisture_volume=False,
    show_ascent_descent_points=False,
    show_convergence_floor=False,
    show_moisture_sheet=False,
    show_divergence_sheet=False,
    show_slice_curtain=False,
    vertical_exaggeration=24.0,
)
jet_context_fig.show()

entrance_circulation_fig = create_3d_case_figure(
    entrance_case_data,
    title=f"Notebook 28 | entrance-region circulation cube | {analysis_time:%Y-%m-%d %H:%M UTC}",
    show_cube_frame=True,
    show_jet_volume=False,
    show_jet_points=True,
    jet_point_threshold=35.0,
    jet_point_size=2.2,
    jet_point_opacity=0.18,
    jet_point_single_color="#475569",
    jet_point_showscale=False,
    max_jet_points=1800,
    show_moisture_volume=False,
    show_ascent_descent_points=False,
    show_convergence_floor=True,
    show_moisture_sheet=False,
    show_divergence_sheet=False,
    show_slice_curtain=True,
    convergence_quantile=0.85,
    convergence_opacity=0.55,
    slice_omega_smoothing_levels=3,
    slice_omega_smoothing_points=9,
    slice_omega_max_abs=0.20,
    slice_omega_opacity=0.60,
    vertical_exaggeration=24.0,
)
entrance_circulation_fig.show()

if exit_case_data is not None:
    exit_circulation_fig = create_3d_case_figure(
        exit_case_data,
        title=f"Notebook 28 | exit-region circulation cube | {analysis_time:%Y-%m-%d %H:%M UTC}",
        show_cube_frame=True,
        show_jet_volume=False,
        show_jet_points=True,
        jet_point_threshold=35.0,
        jet_point_size=2.2,
        jet_point_opacity=0.18,
        jet_point_single_color="#475569",
        jet_point_showscale=False,
        max_jet_points=1800,
        show_moisture_volume=False,
        show_ascent_descent_points=False,
        show_convergence_floor=True,
        show_moisture_sheet=False,
        show_divergence_sheet=False,
        show_slice_curtain=True,
        convergence_quantile=0.85,
        convergence_opacity=0.55,
        slice_omega_smoothing_levels=3,
        slice_omega_smoothing_points=9,
        slice_omega_max_abs=0.20,
        slice_omega_opacity=0.60,
        vertical_exaggeration=24.0,
    )
    exit_circulation_fig.show()
else:
    exit_circulation_fig = None

if SAVE_PLOTS:
    jet_context_path = CASE_PLOT_DIR / f"case27_jet_context_{analysis_time:%Y%m%d_%H%M}.html"
    entrance_path = CASE_PLOT_DIR / f"case27_entrance_circulation_{analysis_time:%Y%m%d_%H%M}.html"
    exit_path = CASE_PLOT_DIR / f"case27_exit_circulation_{analysis_time:%Y%m%d_%H%M}.html"
    jet_context_fig.write_html(jet_context_path, include_plotlyjs="cdn")
    entrance_circulation_fig.write_html(entrance_path, include_plotlyjs="cdn")
    if exit_circulation_fig is not None:
        exit_circulation_fig.write_html(exit_path, include_plotlyjs="cdn")
    print("Saved interactive HTML:", jet_context_path)
    print("Saved interactive HTML:", entrance_path)
    if exit_circulation_fig is not None:
        print("Saved interactive HTML:", exit_path)


## Companion Slice

The 3-D curtains are useful for orientation, but the companion pressure-coordinate section remains the clearest place to inspect ascent, descent, terrain influence, and the vertical stacking of the jet and lower-level forcing. By default it follows the primary jet-relative slice.


In [ ]:
cross_section_runtime = cross_sections_module.build_cross_section_diagnostics(
    get_era5_runtime_ds_27(),
    analysis_time,
    start_lon=float(primary_slice_start[0]),
    start_lat=float(primary_slice_start[1]),
    end_lon=float(primary_slice_end[0]),
    end_lat=float(primary_slice_end[1]),
    terrain_field=terrain_field,
)

pressure_fig = cross_sections_module.plot_pressure_cross_section_figure(
    transect=cross_section_runtime["transect"],
    theta_section=cross_section_runtime["theta_section"],
    omega_section=cross_section_runtime["omega_section"],
    moisture_proxy_section=cross_section_runtime["moisture_proxy_section"],
    pv_section=cross_section_runtime["pv_section"],
    along_wind_section=cross_section_runtime["along_wind_section"],
    zonal_wind_section=cross_section_runtime.get("zonal_wind_section"),
    terrain_pressure_hpa=cross_section_runtime["terrain_pressure_section"],
    terrain_height_m=cross_section_runtime["terrain_height_section"],
    analysis_time=cross_section_runtime["analysis_time"],
    group_id="case27_single_event",
    time_role=f"{primary_slice_label} slice companion",
    wind_render_mode="vectors",
)
display(pressure_fig)

if SAVE_PLOTS:
    pressure_path = CASE_PLOT_DIR / f"case27_pressure_cross_section_{analysis_time:%Y%m%d_%H%M}.png"
    pressure_fig.savefig(pressure_path, dpi=180, bbox_inches="tight")
    print("Saved pressure cross section:", pressure_path)
